<a href="https://colab.research.google.com/github/7vika-lgtm/Skin-classifier/blob/main/dermascan_skin_classifier_colab_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DermaScan: Multiclass Skin Lesion Classification (Google Colab)
**Goal:** train a deep-learning model that classifies a skin image into **7 categories**, then export it to **ONNX** so the DermaScan website can run it privately in the browser.

**Dataset:** DermaMNIST (from MedMNIST v2), a 7-class version of the HAM10000 dermatology dataset (about 10,000 dermoscopy images). It downloads automatically.

| id | class |
|---|---|
| 0 | actinic keratoses and intraepithelial carcinoma |
| 1 | basal cell carcinoma |
| 2 | benign keratosis-like lesions |
| 3 | dermatofibroma |
| 4 | melanoma |
| 5 | melanocytic nevi |
| 6 | vascular lesions |

**Research questions**
1. How well does transfer learning (MobileNetV2) handle 7 imbalanced skin-lesion classes?
2. Which classes are confused, and how well do we catch melanoma?
3. Can the model say "I am not sure"? (confidence thresholds)

> **Important:** HAM10000 contains dermoscopy images with limited skin-tone diversity. A model trained on it is **not reliable on ordinary phone photos** and must never be used to diagnose. Educational project only.

**Tip:** choose **Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [1]:
!pip install -q medmnist onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.


In [2]:
import copy, json, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from medmnist import DermaMNIST, INFO
from sklearn.metrics import (classification_report, confusion_matrix, balanced_accuracy_score,
                             f1_score, recall_score)
sns.set_theme(style="whitegrid")
torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

SIZE = 128                      # input resolution used by the website too
VERSION = f"mobilenetv2-dermamnist-{SIZE}-v1"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics

Using: cuda


## 2. Load the data
Training images get augmentation (flips, rotation, colour jitter). Validation and test images are only resized and normalised.

In [ ]:
labels_dict = INFO["dermamnist"]["label"]
LABELS = [labels_dict[str(i)] for i in range(len(labels_dict))]
NUM_CLASSES = len(LABELS)
print(LABELS)

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
to_int = lambda y: int(y[0])

train_ds = DermaMNIST(split="train", download=True, size=SIZE, transform=train_tf, target_transform=to_int)
val_ds   = DermaMNIST(split="val",   download=True, size=SIZE, transform=eval_tf,  target_transform=to_int)
test_ds  = DermaMNIST(split="test",  download=True, size=SIZE, transform=eval_tf,  target_transform=to_int)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds, batch_size=128, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=128, num_workers=2)
print(len(train_ds), len(val_ds), len(test_ds))

['actinic keratoses and intraepithelial carcinoma', 'basal cell carcinoma', 'benign keratosis-like lesions', 'dermatofibroma', 'melanoma', 'melanocytic nevi', 'vascular lesions']


 24%|██▍       | 88.7M/373M [01:19<03:22, 1.40MB/s]

## 3. Explore: class imbalance

In [ ]:
counts = np.bincount(train_ds.labels.ravel(), minlength=NUM_CLASSES)
plt.figure(figsize=(9, 4))
sns.barplot(x=[l[:22] for l in LABELS], y=counts, color="#1f7a8c")
plt.xticks(rotation=30, ha="right"); plt.title("Training images per class"); plt.ylabel("count"); plt.tight_layout(); plt.show()
print({LABELS[i][:20]: int(c) for i, c in enumerate(counts)})

In [ ]:
raw = DermaMNIST(split="test", download=True, size=SIZE)
fig, axes = plt.subplots(2, 7, figsize=(16, 5))
seen = {}
for img, lab in raw:
    k = int(lab[0])
    if k not in seen: seen[k] = img
    if len(seen) == NUM_CLASSES: break
for k, ax in enumerate(axes[0]): ax.imshow(seen[k]); ax.set_title(LABELS[k][:18], fontsize=8); ax.axis("off")
for ax in axes[1]: ax.axis("off")
plt.suptitle("One example per class"); plt.tight_layout(); plt.show()

**Observation:** melanocytic nevi are about two thirds of the data, while dermatofibroma and vascular lesions are tiny classes. A model that always answers "nevus" would look about 67% accurate yet be useless. So we judge it with **balanced accuracy, macro F1 and per-class recall**, and we use **class weights** in the loss.

## 4. Model: transfer learning with MobileNetV2
MobileNetV2 is small (about 9 MB after export), which makes it practical to run in a web browser.

In [ ]:
weights = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weights)
model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
model = model.to(device)

cw = (counts.sum() / (NUM_CLASSES * counts)) ** 0.5     # softened inverse-frequency weights
crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device), label_smoothing=0.05)
EPOCHS = 12
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(train_loader))
print("Class weights:", np.round(cw, 2))

## 5. Train

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total, ys, ps = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            out = model(x); loss = crit(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        total += loss.item() * len(y)
        ys.append(y.cpu().numpy()); ps.append(out.softmax(1).detach().cpu().numpy())
    return total / len(loader.dataset), np.concatenate(ys), np.concatenate(ps)

hist = {"tr_loss": [], "va_loss": [], "va_bal": [], "va_f1": []}
best_bal, best_state = 0, None
for ep in range(1, EPOCHS + 1):
    tl, _, _ = run_epoch(train_loader, True)
    vl, yv, pv = run_epoch(val_loader, False)
    bal = balanced_accuracy_score(yv, pv.argmax(1)); f1 = f1_score(yv, pv.argmax(1), average="macro")
    for k, v in zip(hist, [tl, vl, bal, f1]): hist[k].append(v)
    if bal > best_bal: best_bal, best_state = bal, copy.deepcopy(model.state_dict())
    print(f"epoch {ep:2d} | train loss {tl:.3f} | val loss {vl:.3f} | val balanced acc {bal:.3f} | val macro-F1 {f1:.3f}")
model.load_state_dict(best_state)
print("Best validation balanced accuracy:", round(best_bal, 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="validation"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist["va_bal"], label="balanced accuracy"); ax[1].plot(hist["va_f1"], label="macro F1"); ax[1].set_title("Validation metrics"); ax[1].legend()
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 6. Evaluate on the test set

In [ ]:
_, y_true, probs = run_epoch(test_loader, False)
y_pred = probs.argmax(1)
bal = balanced_accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Accuracy: {(y_true == y_pred).mean():.3f} | Balanced accuracy: {bal:.3f} | Macro F1: {macro_f1:.3f}\n")
print(classification_report(y_true, y_pred, target_names=[l[:28] for l in LABELS], digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize="true")
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[l[:14] for l in LABELS], yticklabels=[l[:14] for l in LABELS])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Normalised confusion matrix (rows sum to 1)"); plt.tight_layout(); plt.show()

recalls = recall_score(y_true, y_pred, average=None)
plt.figure(figsize=(9, 3.8)); sns.barplot(x=[l[:20] for l in LABELS], y=recalls, color="#c0392b")
plt.xticks(rotation=30, ha="right"); plt.ylim(0, 1); plt.title("Recall per class (how many true cases are caught)"); plt.tight_layout(); plt.show()

In [ ]:
MEL = [i for i, l in enumerate(LABELS) if "melanoma" in l and "nevi" not in l][0]
melanoma_recall = float(recalls[MEL])
print(f"Melanoma recall: {melanoma_recall:.3f}")
off = cm.copy(); np.fill_diagonal(off, 0)
print("\nMost common mistakes (true -> predicted):")
for idx in np.dstack(np.unravel_index(np.argsort(-off.ravel())[:5], off.shape))[0]:
    print(f"  {LABELS[idx[0]][:26]:26s} -> {LABELS[idx[1]][:26]:26s}  {off[idx[0], idx[1]]:.0%} of cases")

**Reading the results:** look at the melanoma row. Melanoma is most often confused with benign nevi, which is exactly the dangerous direction (a missed cancer). This is why the website shows ranked probabilities and always tells users to see a doctor.

## 7. Can the model say "I am not sure"?
If we only answer when confidence is above a threshold, accuracy on the answered cases should rise while coverage falls.

In [ ]:
conf = probs.max(1)
rows = []
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf >= t
    rows.append({"threshold": t, "coverage": keep.mean(), "accuracy on kept": (y_pred[keep] == y_true[keep]).mean() if keep.any() else np.nan})
import pandas as pd
pd.DataFrame(rows).round(3)

In [ ]:
print("Mean confidence when correct:", conf[y_pred == y_true].mean().round(3))
print("Mean confidence when wrong:  ", conf[y_pred != y_true].mean().round(3))

The website uses this idea: results under 60% confidence are shown with a warning that the answer is unreliable.

## 8. Export for the website (ONNX)
The browser runs the model with ONNX Runtime Web. We export the model, check it gives the same answers as PyTorch, and write `model_card.json` with the labels, image size and normalisation values the website needs.

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, SIZE, SIZE)
export_args = dict(input_names=["input"], output_names=["logits"],
                   dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(model, dummy, "skin_model.onnx", dynamo=False, **export_args)
except TypeError:                       # older PyTorch without the dynamo argument
    torch.onnx.export(model, dummy, "skin_model.onnx", **export_args)

import os, onnxruntime as ort
print("Model size:", round(os.path.getsize("skin_model.onnx") / 1e6, 1), "MB")

sess = ort.InferenceSession("skin_model.onnx", providers=["CPUExecutionProvider"])
x = torch.stack([test_ds[i][0] for i in range(8)])
with torch.no_grad():
    torch_out = model(x).numpy()
onnx_out = sess.run(["logits"], {"input": x.numpy()})[0]
diff = float(np.abs(torch_out - onnx_out).max())
print("Max difference PyTorch vs ONNX:", diff)
assert diff < 1e-3, "ONNX output does not match PyTorch"


In [ ]:
attention = [i for i, l in enumerate(LABELS) if "carcinoma" in l or ("melanoma" in l and "nevi" not in l)]
card = {"version": VERSION, "inputSize": SIZE, "mean": MEAN, "std": STD, "labels": LABELS, "attentionClasses": attention,
        "metrics": {"balancedAccuracy": round(float(bal), 4), "macroF1": round(float(macro_f1), 4), "melanomaRecall": round(melanoma_recall, 4)}}
json.dump(card, open("model_card.json", "w"), indent=2)
print(json.dumps(card, indent=2))

In [ ]:
# Download both files, then put them in  client/public/model/  of the website project
try:
    from google.colab import files
    files.download("skin_model.onnx"); files.download("model_card.json")
except ImportError:
    print("Not running in Colab: the files are in the current folder.")

## 9. Conclusions, limitations and ethics
- Write your findings here: balanced accuracy, macro F1, melanoma recall, and the most confused classes.
- **Domain shift:** HAM10000 is dermoscopy. Phone photos look different, so real-world accuracy will be lower than the numbers above.
- **Fairness:** the dataset has few images of darker skin tones. Such a model can be less accurate for those users. Datasets such as Fitzpatrick17k and DDI address this.
- **Scope:** the model only knows 7 lesion types; it cannot recognise acne, rashes or eczema.
- **Safety:** this tool should only ever be framed as educational. Real decisions need a clinician.

## Next steps
- Larger input size (224) or a stronger backbone (EfficientNet, ConvNeXt)
- Grad-CAM heatmaps to check the model looks at the lesion
- Temperature scaling to calibrate confidence
- Train on more diverse, phone-photo datasets

## References
- Yang et al., *MedMNIST v2*, Scientific Data (2023).
- Tschandl et al., *The HAM10000 dataset*, Scientific Data (2018).
- Sandler et al., *MobileNetV2: Inverted Residuals and Linear Bottlenecks* (2018).